In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

tezheng = ['ggt', 'drinking', 'ast', 'age', 'hr', 'alt', 'mono', 'gender',
           'umalb', 'sua', 'dbil', 'bun', 'tp', 'idbil', 'rbc']
df = pd.read_csv('../数据/训练集.csv')
print(f"数据形状: {df.shape}，正样本比例: {df['ms_cds'].mean():.3f}")

X = df[tezheng].values
y = df['ms_cds'].values

default_params = {
    'n_estimators': 514,
    'learning_rate': 0.11483790,
    'max_depth': 14,
    'min_child_weight': 1,
    'subsample': 0.86841410,
    'colsample_bytree': 0.41070400,
    'reg_alpha': 0.12015366,
    'reg_lambda': 0.00028341,
    'gamma': 3.45672074e-06,
    'random_state': 42,
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'verbosity': 0,
    'n_jobs': 1
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid_config = {
    'n_estimators': (10, 550, 10),
    'subsample':    (0.775, 0.9,10)
}

param_grids = {}
for p, (start, end, num) in param_grid_config.items():
    if p == 'n_estimators':
        values = np.unique(np.linspace(start, end, num, dtype=int))
    else:
        values = np.linspace(start, end, num)
    param_grids[p] = values.tolist()
    print(f"{p}: {param_grids[p]}")

results = {}

for param_name, param_values in param_grids.items():
    print(f"\n正在处理参数: {param_name} ...")
    train_means, train_stds, cv_means, cv_stds = [], [], [], []

    for value in param_values:
        params = default_params.copy()
        params[param_name] = value

        train_scores = []
        test_scores = []

        for train_idx, val_idx in cv.split(X, y):
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]

            model = xgb.XGBClassifier(**params)
            model.fit(X_tr, y_tr)

            # 训练集 AUC
            y_tr_pred = model.predict_proba(X_tr)[:, 1]
            train_auc = roc_auc_score(y_tr, y_tr_pred)
            train_scores.append(train_auc)

            # 验证集 AUC
            y_val_pred = model.predict_proba(X_val)[:, 1]
            val_auc = roc_auc_score(y_val, y_val_pred)
            test_scores.append(val_auc)

        train_scores = np.array(train_scores)
        test_scores = np.array(test_scores)
        # 过滤可能的 NaN
        if np.isnan(train_scores).any() or np.isnan(test_scores).any():
            train_scores = train_scores[~np.isnan(train_scores)]
            test_scores = test_scores[~np.isnan(test_scores)]

        train_means.append(np.mean(train_scores))
        train_stds.append(np.std(train_scores))
        cv_means.append(np.mean(test_scores))
        cv_stds.append(np.std(test_scores))

    results[param_name] = {
        'values': param_values,
        'train_mean': train_means,
        'train_std': train_stds,
        'cv_mean': cv_means,
        'cv_std': cv_stds
    }

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.linestyle'] = '-'
plt.rcParams['grid.alpha'] = 0.7
plt.rcParams['figure.dpi'] = 300

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (param_name, res) in enumerate(results.items()):
    ax = axes[idx]
    values = np.array(res['values'])
    train_mean = np.array(res['train_mean'])
    train_std = np.array(res['train_std'])
    cv_mean = np.array(res['cv_mean'])
    cv_std = np.array(res['cv_std'])

    # 红线：训练集
    ax.plot(values, train_mean, 'r-s', linewidth=2, markersize=8, label='Training score')
    ax.fill_between(values,
                    train_mean - train_std,
                    train_mean + train_std,
                    alpha=0.2, color='red')

    # 绿线：验证集
    ax.plot(values, cv_mean, 'g-o', linewidth=2, markersize=8, label='Cross-validation score')
    ax.fill_between(values,
                    cv_mean - cv_std,
                    cv_mean + cv_std,
                    alpha=0.2, color='green')

    ax.set_title(f'Validation Curve with {param_name}', fontsize=18)
    ax.set_xlabel(param_name, fontsize=14)
    ax.set_ylabel('Score (AUC)', fontsize=14)
    ax.set_ylim(0.7, 1.001)
    ax.legend(fontsize=14)
    ax.ticklabel_format(axis='x', style='plain')

plt.tight_layout()
plt.savefig('validation_curve_nestimators_subsample.png', dpi=300, bbox_inches='tight')
plt.show()